
## Collecting Data from API

Opening conda prompt:

1. cd /d D:\crypto_trading

2. conda activate cryptotrading

3. jupyter notebook

https://developers.binance.com/docs/binance-spot-api-docs

https://www.bitmex.com/app/apiOverview

In [1]:
# import necessary files
import logging
from exchanges.binance import BinanceClient
from exchanges.coinbase import CoinbaseClient
from database import Hdf5Client
from utils.time_convertor import *
import pandas as pd
import numpy as np
import time

#strategies
import strategies.obv
import strategies.ichimoku as ic

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 5000)
pd.set_option('display.expand_frame_repr', False)  # prevent wrapping of DataFrame

In [3]:
h5_db= Hdf5Client("Binance")
data= h5_db.get_data("BTCUSDT", from_time= 0, to_time= int(time.time() * 1000))


In [4]:
#import and format the data
data= resample_timeframe(data, "15m")
data

,open,high,low,close,volume
timestamp,,,,,
2025-03-26 06:15:00,87491.02,87587.24,87491.03,87565.05,60.468010
2025-03-26 06:30:00,87565.04,87768.00,87565.05,87767.99,184.820200
2025-03-26 06:45:00,87729.25,87923.92,87729.25,87909.07,166.300280
2025-03-26 07:00:00,87881.90,87953.63,87774.36,87841.12,212.873040
2025-03-26 07:15:00,87841.11,88030.46,87786.69,88009.79,137.899660
2025-03-26 07:30:00,87972.22,88259.65,87972.22,88211.29,464.476420
2025-03-26 07:45:00,88186.22,88275.00,88145.87,88140.45,238.222820
2025-03-26 08:00:00,88132.30,88200.00,88027.05,88053.83,146.987880
2025-03-26 08:15:00,88053.83,88149.79,88004.96,88041.26,113.948560


## Testing out strategies: 

##### Ichimoku:

In [5]:
data= ic.backtest(data, 9, 26)
data

,open,high,low,close,volume,tenkan_sen,kijun_sen,senkou_span_a,senkou_span_b,chikou_span,tenkan_minus_kijun,prev_tenkan_minus_kijun,signal,pnl
timestamp,,,,,,,,,,,,,,
2025-03-27 10:30:00,87507.68,87550.00,87435.01,87425.12,68.13910,87465.815,87430.915,87357.3675,86816.025,86961.76,34.900,-12.530,1,NaN
2025-03-30 08:45:00,83265.01,83406.14,83265.02,83378.09,81.20280,83239.595,83186.245,82820.4075,82460.515,82834.39,53.350,-78.525,1,-0.046291
2025-03-31 00:30:00,81927.68,81992.78,81408.00,81361.51,471.41848,82082.850,82158.055,82550.8625,82797.170,82071.99,-75.205,55.685,-1,-0.024186
2025-03-31 09:30:00,81490.12,81577.82,81376.01,81496.80,287.99958,81819.875,81840.790,81976.9125,82200.940,83339.99,-20.915,2.845,-1,-0.001663
2025-04-04 07:00:00,83054.72,83567.00,83054.73,83522.84,393.02241,83108.720,83091.180,82727.7325,82365.140,82329.99,17.540,-57.110,1,-0.024860
2025-04-05 03:00:00,83957.88,83976.61,83819.42,83822.87,123.74655,84026.800,83961.780,83618.3475,83231.580,83783.02,65.020,-73.205,1,0.003592
2025-04-11 00:15:00,79630.48,79722.78,79506.29,79454.96,163.43161,79667.615,79669.505,79790.5825,80485.170,81102.00,-1.890,46.835,-1,-0.052109
2025-04-13 01:00:00,85466.82,85800.00,85444.10,85777.71,195.06800,85416.515,85369.875,84626.1425,84348.590,84461.28,46.640,-67.800,1,-0.079577
2025-04-13 12:15:00,84577.79,84700.00,84273.99,84283.01,223.30036,84552.435,84611.925,85162.9175,85256.720,84730.00,-59.490,17.810,-1,-0.017425


In [6]:
print(data.columns)

Index(['open', 'high', 'low', 'close', 'volume', 'tenkan_sen', 'kijun_sen', 'senkou_span_a', 'senkou_span_b', 'chikou_span', 'tenkan_minus_kijun', 'prev_tenkan_minus_kijun', 'signal', 'pnl'], dtype='object')


In [10]:
import plotly.graph_objects as go
import plotly.offline as pyo
fig= go.Figure()

In [13]:
# plot ohlc candles for btcusdt

fig.add_trace(go.Candlestick(
    x= data.index,
    open= data["open"],
    high= data["high"],
    low= data["low"],
    close= data["close"],
    name= "BTCUSDT OHLC"))

#Ichimoku Lines
fig.add_trace(go.Scatter(x=data.index, y=data['tenkan_sen'], line=dict(color='blue'), name='Tenkan Sen'))
fig.add_trace(go.Scatter(x=data.index, y=data['kijun_sen'], line=dict(color='purple'), name='Kijun Sen'))
fig.add_trace(go.Scatter(x=data.index, y=data['senkou_span_a'], line=dict(color='green'), name='Senkou Span A', fill=None))
fig.add_trace(go.Scatter(x=data.index, y=data['senkou_span_b'], line=dict(color='red'), name='Senkou Span B', fill='tonexty', fillcolor='rgba(50,200,50,0.2)'))
fig.add_trace(go.Scatter(x=data.index, y=data['chikou_span'], line=dict(color='orange'), name='Chikou Span'))

# Mark Buy Signals
buy_signals = data[data['signal'] == 1]
fig.add_trace(go.Scatter(mode='markers', x=buy_signals.index, y=buy_signals['close'],
                         marker=dict(color='green', size=10, symbol='triangle-up'),
                         name='Buy Signal'))

# Mark Sell Signals
sell_signals = data[data['signal'] == -1]
fig.add_trace(go.Scatter(mode='markers', x=sell_signals.index, y=sell_signals['close'],
                         marker=dict(color='red', size=10, symbol='triangle-down'),
                         name='Sell Signal'))


# Update Layout
fig.update_layout(
    title="Ichimoku Trading Strategy Visualization",
    xaxis_title="Date",
    yaxis_title="Price",
    legend_title="Legend",
    width=1200,
    height=600
)

pyo.plot(fig, filename= "ichimoku.html")

'ichimoku.html'